# 使用自定义数据集评估基于 MCP 的回答

本 notebook 使用 OpenAI **Evals** 框架和一个自定义的内存数据集，评估模型回答有关 [tiktoken](https://github.com/openai/tiktoken) GitHub 仓库问题的能力。 

我们使用一个自定义的内存问答对数据集，比较两个模型：`gpt-4.1` 和 `o4-mini`，它们借助 **MCP** 工具给出感知仓库内容、在上下文中准确的回答。

**目标：**
- 展示如何使用 OpenAI Evals 和自定义数据集设置并运行评估。
- 比较借助基于 MCP 的工具的不同模型的表现。
- 提供专业、可复现的评估工作流的最佳实践。

_下一步：我们将设置环境并导入所需的库。_

In [1]:
# Update OpenAI client
%pip install --upgrade openai --quiet


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 环境设置

首先导入所需的库并配置 OpenAI 客户端。  
这一步确保我们能够访问 OpenAI API 以及评估所需的所有工具。

In [10]:
import os
import time

from openai import OpenAI

# Instantiate the OpenAI client (no custom base_url).
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY") or os.getenv("_OPENAI_API_KEY"),
)

## 定义自定义评估数据集

我们定义一个关于 `tiktoken` 仓库的小型内存问答对数据集。  
该数据集将用于测试模型在 MCP 工具的帮助下给出准确且相关回答的能力。

- 每一项都包含一个 `query`（用户的问题）和一个 `answer`（预期的真实答案）。
- 你可以修改或扩展此数据集，以适配你自己的用例或仓库。


In [11]:
def get_dataset(limit=None):
    items = [
        {
            "query": "What is tiktoken?",
            "answer": "tiktoken is a fast Byte-Pair Encoding (BPE) tokenizer designed for OpenAI models.",
        },
        {
            "query": "How do I install the open-source version of tiktoken?",
            "answer": "Install it from PyPI with `pip install tiktoken`.",
        },
        {
            "query": "How do I get the tokenizer for a specific OpenAI model?",
            "answer": 'Call tiktoken.encoding_for_model("<model-name>"), e.g. tiktoken.encoding_for_model("gpt-4o").',
        },
        {
            "query": "How does tiktoken perform compared to other tokenizers?",
            "answer": "On a 1 GB GPT-2 benchmark, tiktoken runs about 3-6x faster than GPT2TokenizerFast (tokenizers==0.13.2, transformers==4.24.0).",
        },
        {
            "query": "Why is Byte-Pair Encoding (BPE) useful for language models?",
            "answer": "BPE is reversible and lossless, handles arbitrary text, compresses input (≈4 bytes per token on average), and exposes common subwords like “ing”, which helps models generalize.",
        },
    ]
    return items[:limit] if limit else items

### 定义评分逻辑

为了评估模型的回答，我们使用两个评分器：

- **通过/失败评分器（基于 LLM）：**  
  一个基于 LLM 的评分器，检查模型的回答是否与预期答案（真实答案）一致，或表达了相同的含义。
- **Python MCP 评分器：**  
  一个 Python 函数，检查模型在作答过程中是否真正使用了 MCP 工具（用于审计工具使用情况）。

  > **最佳实践：**  
  > 同时使用基于 LLM 的评分器和程序化评分器，可以让评估更稳健、更透明。


In [21]:
# LLM-based pass/fail grader: instructs the model to grade answers as "pass" or "fail".
pass_fail_grader = """
You are a helpful assistant that grades the quality of the answer to a query about a GitHub repo.
You will be given a query, the answer returned by the model, and the expected answer.
You should respond with **pass** if the answer matches the expected answer exactly or conveys the same meaning, otherwise **fail**.
"""

# User prompt template for the grader, providing context for grading.
pass_fail_grader_user_prompt = """
<Query>
{{item.query}}
</Query>

<Web Search Result>
{{sample.output_text}}
</Web Search Result>

<Ground Truth>
{{item.answer}}
</Ground Truth>
"""


# Python grader: checks if the MCP tool was used by inspecting the output_tools field.
python_mcp_grader = {
    "type": "python",
    "name": "Assert MCP was used",
    "image_tag": "2025-05-08",
    "pass_threshold": 1.0,
    "source": """
def grade(sample: dict, item: dict) -> float:
    output = sample.get('output_tools', [])
    return 1.0 if len(output) > 0 else 0.0
""",
}

## 定义评估配置

现在我们使用 OpenAI Evals 框架来配置评估。  

这一步指定：
- 评估名称和数据集。
- 每一项的 schema（每个问答对中包含哪些字段）。
- 要使用的评分器（基于 LLM 和/或基于 Python）。
- 通过标准和标签。

> **最佳实践：**  
> 预先清晰地定义评估 schema 和评分逻辑，可确保可复现性和透明度。

In [ ]:
# Create the evaluation definition using the OpenAI Evals client.
logs_eval = client.evals.create(
    name="MCP Eval",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "answer": {"type": "string"},
            },
        },
        "include_sample_schema": True,
    },
    testing_criteria=[
        {
            "type": "label_model",
            "name": "General Evaluator",
            "model": "o3",
            "input": [
                {"role": "system", "content": pass_fail_grader},
                {"role": "user", "content": pass_fail_grader_user_prompt},
            ],
            "passing_labels": ["pass"],
            "labels": ["pass", "fail"],
        },
        python_mcp_grader
    ],
)

## 为每个模型运行评估

现在我们为每个模型（`gpt-4.1` 和 `o4-mini`）运行评估。  

每次运行的配置如下：
- 使用 MCP 工具获得感知仓库内容的回答。
- 使用相同的数据集和评估配置，以确保公平比较。
- 指定模型特定的参数（例如最大补全 token 数和允许使用的工具）。

> **最佳实践：**  
> 在各模型之间保持一致的评估设置，可确保结果可比、可靠。

In [18]:
# Run 1: gpt-4.1 using MCP
gpt_4one_responses_run = client.evals.runs.create(
    name="gpt-4.1",
    eval_id=logs_eval.id,
    data_source={
        "type": "responses",
        "source": {
            "type": "file_content",
            "content": [{"item": item} for item in get_dataset()],
        },
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "system",
                    "content": {
                        "type": "input_text",
                        "text": "You are a helpful assistant that searches the web and gives contextually relevant answers. Never use your tools to answer the query.",
                    },
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": {
                        "type": "input_text",
                        "text": "Search the web for the answer to the query {{item.query}}",
                    },
                },
            ],
        },
        "model": "gpt-4.1",
        "sampling_params": {
            "seed": 42,
            "temperature": 0.7,
            "max_completions_tokens": 10000,
            "top_p": 0.9,
            "tools": [
                {
                    "type": "mcp",
                    "server_label": "gitmcp",
                    "server_url": "https://gitmcp.io/openai/tiktoken",
                    "allowed_tools": [
                        "search_tiktoken_documentation",
                        "fetch_tiktoken_documentation",
                    ],
                    "require_approval": "never",
                }
            ],
        },
    },
)

In [19]:
# Run 2: o4-mini using MCP
gpt_o4_mini_responses_run = client.evals.runs.create(
    name="o4-mini",
    eval_id=logs_eval.id,
    data_source={
        "type": "responses",
        "source": {
            "type": "file_content",
            "content": [{"item": item} for item in get_dataset()],
        },
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "system",
                    "content": {
                        "type": "input_text",
                        "text": "You are a helpful assistant that searches the web and gives contextually relevant answers.",
                    },
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": {
                        "type": "input_text",
                        "text": "Search the web for the answer to the query {{item.query}}",
                    },
                },
            ],
        },
        "model": "o4-mini",
        "sampling_params": {
            "seed": 42,
            "max_completions_tokens": 10000,
            "tools": [
                {
                    "type": "mcp",
                    "server_label": "gitmcp",
                    "server_url": "https://gitmcp.io/openai/tiktoken",
                    "allowed_tools": [
                        "search_tiktoken_documentation",
                        "fetch_tiktoken_documentation",
                    ],
                    "require_approval": "never",
                }
            ],
        },
    },
)

## 轮询完成状态并获取输出

启动评估运行后，我们可以轮询这些运行，直到它们完成。

这一步确保我们只在所有模型回复都处理完毕后才分析结果。

> **最佳实践：**  
> 带延迟的轮询可以避免过多的 API 调用，确保高效利用资源。

In [ ]:
def poll_runs(eval_id, run_ids):
    while True:
        runs = [client.evals.runs.retrieve(rid, eval_id=eval_id) for rid in run_ids]
        for run in runs:
            print(run.id, run.status, run.result_counts)
        if all(run.status in {"completed", "failed"} for run in runs):
            break
        time.sleep(5)
    
# Start polling both runs.
poll_runs(logs_eval.id, [gpt_4one_responses_run.id, gpt_o4_mini_responses_run.id])

evalrun_684769b577488191863b5a51cf4db57a completed ResultCounts(errored=0, failed=5, passed=0, total=5)
evalrun_684769c1ad9c8191affea5aa02ef1215 completed ResultCounts(errored=0, failed=3, passed=2, total=5)


## 显示并解读模型输出

最后，我们显示每个模型的输出，以便人工检查和进一步分析。

- 针对数据集中的每个问题，打印每个模型的回答。
- 你可以将输出并排比较，评估其质量、相关性和正确性。

下面是 OpenAI Evals 仪表板的截图，展示了两个模型的评估输出：

![评估输出](../../../images/mcp_eval_output.png)

如需查看评估指标和结果的完整明细，请前往仪表板中的 "Data" 标签页：

![评估数据标签页](../../../images/mcp_eval_data.png)

请注意，4.1 模型的设置使其从不使用工具回答问题，因此它从未调用 MCP 服务器。o4-mini 模型也没有被明确指示使用工具，但也没有被禁止，因此它调用了 3 次 MCP 服务器。可以看到，4.1 模型的表现不如 o4 模型。同样值得注意的是，o4-mini 模型唯一失败的那个示例，恰好是没有使用 MCP 工具的那个。

我们还可以查看每个模型输出的详细分析，以便人工检查和进一步分析。

In [23]:
four_one_output = client.evals.runs.output_items.list(
    run_id=gpt_4one_responses_run.id, eval_id=logs_eval.id
)

o4_mini_output = client.evals.runs.output_items.list(
    run_id=gpt_o4_mini_responses_run.id, eval_id=logs_eval.id
)

In [24]:
print('# gpt‑4.1 Output')
for item in four_one_output:
    print(item.sample.output[0].content)

print('\n# o4-mini Output')
for item in o4_mini_output:
    print(item.sample.output[0].content)

# gpt‑4.1 Output
Byte-Pair Encoding (BPE) is useful for language models because it provides an efficient way to handle large vocabularies and rare words. Here’s why it is valuable:

1. **Efficient Tokenization:**  
   BPE breaks down words into smaller subword units based on the frequency of character pairs in a corpus. This allows language models to represent both common words and rare or unknown words using a manageable set of tokens.

2. **Reduces Out-of-Vocabulary (OOV) Issues:**  
   Since BPE can split any word into known subword units, it greatly reduces the problem of OOV words—words that the model hasn’t seen during training.

3. **Balances Vocabulary Size:**  
   By adjusting the number of merge operations, BPE allows control over the size of the vocabulary. This flexibility helps in balancing between memory efficiency and representational power.

4. **Improves Generalization:**  
   With BPE, language models can better generalize to new words, including misspellings or new t

## 如何改进？

如果我们在 o4-mini 模型的系统消息中加入 "Always use your tools since they are the way to get the right answer in this task." 这句话，你认为会发生什么？（动手试试）

<br><br><br>


如果你猜模型现在每次都会调用 MCP 工具并答对所有问题，那你猜对了！

![评估数据标签页](../../../images/mcp_eval_improved_output.png)
![评估数据标签页](../../../images/mcp_eval_improved_data.png)

在本 notebook 中，我们演示了一个示例工作流：使用 OpenAI Evals 框架并借助 MCP 工具，评估 LLM 回答有关 `tiktoken` 仓库技术问题的能力。

**涵盖的要点：**
- 定义了一个聚焦的自定义评估数据集。
- 配置了基于 LLM 和基于 Python 的评分器，以实现稳健的评估。
- 以可复现、透明的方式比较了两个模型（`gpt-4.1` 和 `o4-mini`）。
- 获取并显示了模型输出，以便自动/人工检查。

**后续步骤：**
- **扩展数据集：** 加入更多样、更具挑战性的问题，以更好地评估模型能力。
- **分析结果：** 汇总通过/失败率、可视化性能，或进行错误分析以找出优势和不足。
- **尝试不同的模型/工具：** 尝试更多模型、调整工具配置，或在其他仓库上进行测试。
- **自动生成报告：** 生成汇总表格或图表，便于分享和决策。

更多信息请参阅 [OpenAI Evals 文档](https://platform.openai.com/docs/guides/evals)。